COURSEWORK 1 AI in Robotics - UNO Machine Vision

In [ ]:
!pip install tensorflow pandas scikit-learn opencv-python

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input,
    Conv2D,
    MaxPooling2D,
    GlobalAveragePooling2D,
    Flatten,
    Dense,
    Dropout,
    BatchNormalization,
    Rescaling,
    RandomRotation,
    RandomZoom,
    RandomBrightness,
    RandomContrast,
    RandomFlip
)
from tensorflow.keras.optimizers import Adam
import pandas as pd
import numpy as np
import cv2
import os
from matplotlib import pyplot as plt

In [ ]:
# Load file from Drive
from google.colab import drive
drive.mount('/content/drive/')

In [ ]:
data_dir = 'drive/MyDrive/Uno Dataset/'

In [ ]:
batch_size = 2
img_height = 224
img_width = 224

In [ ]:
full_ds = tf.keras.utils.image_dataset_from_directory(
  data_dir,
  shuffle=True,
  seed=123,
  image_size=(img_height, img_width),
  batch_size=1)

In [ ]:
# Calculate dataset size and split
dataset_size = full_ds.cardinality().numpy()
train_size = int(0.7 * dataset_size)  # 70% training
val_size = int(0.15 * dataset_size)   # 15% validation
test_size = dataset_size - train_size - val_size

In [ ]:
# Split into train, validation, and test sets
train_ds = full_ds.take(train_size)
val_ds = full_ds.skip(train_size).take(val_size)
test_ds = full_ds.skip(train_size + val_size)

In [ ]:
class_names = full_ds.class_names
class_names

In [ ]:
normalization_layer = Rescaling(1./255)
normalized_ds = train_ds.map (lambda x,y : (normalization_layer(x), y))
normalized_ds

In [ ]:
normalized_val_ds = val_ds.map(lambda x,y : (normalization_layer(x), y))

In [ ]:
normalized_test_ds = test_ds.map(lambda x,y : (normalization_layer(x), y))

In [ ]:
data_augmentation = tf.keras.Sequential([
    RandomRotation(0.05),
    RandomBrightness(0.05),
    RandomZoom(0.05,0.05),
    RandomFlip("horizontal")
])

In [ ]:
# Augment the training dataset
augmented_train_dataset = normalized_ds.map(
    lambda x, y: (data_augmentation(x, training=True), y)
)

In [ ]:
combined_train_dataset = normalized_ds.concatenate(augmented_train_dataset)


In [ ]:
# Build The Model
input_shape = (224, 224, 3)
inputs = Input(shape=input_shape)

#aug_layer = data_augmentation(inputs)

features_extraction = Conv2D(32, 3, padding="valid", activation="relu")(inputs)
features_extraction = MaxPooling2D((2, 2))(features_extraction)
features_extraction = Conv2D(32, 3, padding="valid", activation="relu")(features_extraction)
features_extraction = MaxPooling2D((2, 2))(features_extraction)
features_extraction = Conv2D(64, 3, padding="valid", activation="relu")(features_extraction)
features_extraction = MaxPooling2D((2, 2))(features_extraction)
features_extraction = Conv2D(64, 3, padding="valid", activation="relu")(features_extraction)
features_extraction = MaxPooling2D((2, 2))(features_extraction)
features_extraction = Conv2D(128, 3, padding="same", activation="relu")(features_extraction)
features_extraction = MaxPooling2D((2, 2))(features_extraction)
features_extraction = Conv2D(256, 3, padding="same", activation="relu")(features_extraction)
features_extraction = MaxPooling2D((2, 2))(features_extraction)
features_extraction = Conv2D(512, 3, padding="same", activation="relu")(features_extraction)
features_extraction = MaxPooling2D((2, 2))(features_extraction)
features_extraction = Conv2D(512, 3, padding="same", activation="relu")(features_extraction)



x = GlobalAveragePooling2D()(features_extraction)
x = Dense(1024, activation='relu')(x)
x = Dropout(0.2)(x)
x = Dense(1024, activation='relu')(x)
x = Dropout(0.2)(x)

outputs = Dense(54, activation='softmax', name='value_output')(x)


model = Model(inputs=inputs, outputs=outputs)

model.compile(
    optimizer=Adam(0.0001),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"]
)

model.summary()

In [ ]:
# Train Model
epochs = 10


In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

In [ ]:
# Implementation of Early Stopping to prevent overfitting
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=7,              # Number of epochs with no improvement after which training will be stopped
    verbose=1,               # Verbosity mode
    restore_best_weights=True # Restore model weights from the epoch with the best value of the monitored metric
)

In [ ]:
model_checkpoint = ModelCheckpoint(
    filepath='best_model.keras',    # File path where the model will be saved
    monitor='val_loss',          # Metric to monitor
    save_best_only=True,         # Saves the model only when the monitored metric improves
    verbose=1
)

In [ ]:
history = model.fit(
    combined_train_dataset,
    validation_data=normalized_val_ds,
    shuffle=True,
    epochs=epochs,
)

In [ ]:
history2 = model.fit(
    combined_train_dataset,
    validation_data=normalized_val_ds,
    shuffle=True,
    epochs=10,
)

In [ ]:
# Testing
model.evaluate(normalized_test_ds)

In [ ]:
normalized_test_ds

In [ ]:
model.save('model2.keras')

In [ ]:
model.save('model2.h5')

In [ ]:
from tensorflow.keras.preprocessing import image

img = image.load_img("IMG_3180.JPG",target_size=(224,224))

# Convert the image to an array
img_array = image.img_to_array(img)

# Expand dimensions to match the model’s input shape (batch size of 1)
img_array = np.expand_dims(img_array, axis=0)

# Optionally, scale/normalize the image (depends on your model’s training preprocessing)
img_array = img_array / 255.0

# Predict
predictions = model.predict(img_array)

# Optionally, decode the predictions
predicted_class = np.argmax(predictions, axis=1)
confidence = np.max(predictions)


print(class_names[predicted_class[0]], confidence)

In [ ]:
print(f'Test Value Accuracy: {test_value_accuracy}')
print(f'Test Color Accuracy: {test_color_accuracy}')